# VisionBridge A-Z base model training (error-resistant Colab)

Run these cells **from top to bottom**.

What this notebook does:
1. Creates an isolated **Python 3.12** environment with `uv`
2. Installs PyTorch (CUDA if Colab GPU is available), NumPy 2.1.3, MediaPipe 0.10.35
3. Downloads the MediaPipe Hand Landmarker model
4. Materializes the repository-local RealSign Git-LFS dataset archive and extracts it
5. Extracts 126D two-hand landmarks (stratified 80/20 train/val from train+val pools)
6. Verifies the active V3 architecture contract (126D -> 128D -> 64D -> 26 classes)
7. Trains the letter base model (up to 500 epochs) and saves the best checkpoint

The original RealSign **test** split is left untouched and evaluated after training.

**Tips**
- Use a **GPU** runtime (Runtime → Change runtime type → GPU) for faster training.
- If a cell fails, read the printed stderr — this notebook surfaces subprocess output.
- Re-run from the top after a runtime restart.

In [ ]:
# Cell 1 — Environment setup (isolated Python 3.12 + deps)
from pathlib import Path
import os
import shutil
import subprocess
import sys

ROOT = Path("/content")
os.chdir(ROOT)
REPO = ROOT / "VisionBridge"
VENV = ROOT / "visionbridge_train_env"
PYTHON = VENV / "bin" / "python"


def run(cmd, env=None, check=True, cwd=None):
    """Run a command and always stream stdout/stderr so Colab shows the real error."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    result = subprocess.run(
        [str(c) for c in cmd],
        check=False,
        env=env,
        cwd=cwd,
        text=True,
        capture_output=True,
    )
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n", file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}:\n"
            f"  {' '.join(str(c) for c in cmd)}\n"
            f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
        )
    return result


for path in (REPO, VENV):
    if path.exists():
        print(f"Removing previous {path} ...")
        shutil.rmtree(path)

print("Cloning VisionBridge without automatic LFS smudge...")
clone_env = os.environ.copy()
clone_env["GIT_LFS_SKIP_SMUDGE"] = "1"
run([
    "git", "clone", "--depth", "1",
    "https://github.com/BharathWaj-K-R/VisionBridge.git",
    str(REPO),
], env=clone_env)

print("Ensuring Git LFS is installed...")
if shutil.which("git-lfs") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "git-lfs"])
run(["git", "lfs", "install", "--local"], cwd=REPO)

print("Materializing repository-local RealSign dataset...")
run([
    "git", "lfs", "pull", "--include", "data/raw/RealSign/Dataset.zip",
], cwd=REPO)

print("Installing uv...")
run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--disable-pip-version-check", "uv",
])

UV = [sys.executable, "-m", "uv"]

# Ensure a managed CPython 3.12 is available (Colab may ship 3.11/3.12/3.13)
print("Ensuring Python 3.12 is available via uv...")
run([*UV, "python", "install", "3.12"])

print("Creating isolated venv at", VENV)
run([*UV, "venv", "--python", "3.12", str(VENV)])

# Detect whether Colab has a GPU so we can install a matching torch wheel
gpu_probe = subprocess.run(
    ["nvidia-smi"], capture_output=True, text=True
)
has_gpu = gpu_probe.returncode == 0
print("GPU available (nvidia-smi):", has_gpu)

# Keep the notebook reproducible. Use the exact CPU build used by the
# repository on CPU-only runtimes and the matching upstream PyTorch release
# on Colab GPU runtimes.

install_cmd = [
    *UV,
    "pip",
    "install",
    "--python",
    str(PYTHON),
    "numpy==2.1.3",
    "mediapipe==0.10.35",
]
if has_gpu:
    print("Installing PyTorch 2.9.0 for the Colab GPU...")
    install_cmd.append("torch==2.9.0")
else:
    print("Installing PyTorch 2.9.0 CPU build...")
    install_cmd.extend([
        "--extra-index-url",
        "https://download.pytorch.org/whl/cpu",
        "torch==2.9.0+cpu",
    ])

run(install_cmd)

# Colab sets MPLBACKEND=module://matplotlib_inline.backend_inline which is NOT
# installed inside the isolated venv. MediaPipe imports matplotlib on load, so
# force a headless backend for every venv subprocess.
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "backend")
env["MPLBACKEND"] = "Agg"
env.pop("MPLCONFIGDIR", None)

smoke = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
import sys
import torch
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("MediaPipe:", mp.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

assert sys.version_info[:2] == (3, 12), sys.version_info
assert np.__version__ == "2.1.3", np.__version__
assert mp.__version__ == "0.10.35", mp.__version__

print("MediaPipe Tasks API:", vision.HandLandmarker.__name__)
print("Training environment: PASS")
'''

run([str(PYTHON), "-c", smoke], env=env)
print("Cell 1 complete.")



In [ ]:
# Cell 2 — Download and verify MediaPipe Hand Landmarker .task model
from pathlib import Path
import hashlib
import urllib.request

HAND_MODEL_URL = (
    "https://storage.googleapis.com/mediapipe-models/"
    "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
)
HAND_MODEL_PATH = Path("/content/hand_landmarker.task")
HAND_MODEL_SIZE = 7_819_105
HAND_MODEL_SHA256 = "fbc2a30080c3c557093b5ddfc334698132eb341044ccee322ccf8bcf3607cde1"

if HAND_MODEL_PATH.exists():
    HAND_MODEL_PATH.unlink()

print("Downloading MediaPipe Hand Landmarker model...")
urllib.request.urlretrieve(HAND_MODEL_URL, HAND_MODEL_PATH)

size = HAND_MODEL_PATH.stat().st_size
digest = hashlib.sha256(HAND_MODEL_PATH.read_bytes()).hexdigest()
if size != HAND_MODEL_SIZE or digest != HAND_MODEL_SHA256:
    raise RuntimeError(
        f"Hand model integrity check failed: size={size}, sha256={digest}"
    )

print("Hand model bytes:", size)
print("Hand model SHA-256:", digest)
print("Hand model download: PASS")


In [ ]:
# Cell 3 — Verify Hand Landmarker can load in the training venv
import os
import subprocess
import sys
from pathlib import Path

PYTHON = Path("/content/visionbridge_train_env/bin/python")
MODEL = Path("/content/hand_landmarker.task")
ENV = os.environ.copy()
ENV["MPLBACKEND"] = "Agg"

smoke = r'''
import os
os.environ["MPLBACKEND"] = "Agg"
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

model_path = __import__("sys").argv[1]
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
)
detector = vision.HandLandmarker.create_from_options(options)
detector.close()
print("Hand Landmarker initialization: PASS")
'''

result = subprocess.run(
    [str(PYTHON), "-c", smoke, str(MODEL)],
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Hand Landmarker smoke test failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

In [ ]:
# Cell 4 — Verify and extract the repository-local RealSign dataset archive
from pathlib import Path
import hashlib
import shutil
import subprocess
import sys
import zipfile

REPO = Path("/content/VisionBridge")
ZIP_PATH = REPO / "data/raw/RealSign/Dataset.zip"
DATASET_DIR = Path("/content/RealSign")
EXPECTED_SIZE = 656_689_688
EXPECTED_SHA256 = "008cae248e346b8c31fbbea057fcc3f69c6909d29a88e6bb1fb0369f528de2b5"

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

if not ZIP_PATH.is_file():
    raise RuntimeError("Repository-local RealSign archive is missing. Git LFS materialization failed.")

size = ZIP_PATH.stat().st_size
digest = hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest()
if size != EXPECTED_SIZE or digest != EXPECTED_SHA256:
    raise RuntimeError(
        f"Repository-local RealSign archive integrity failed: size={size}, sha256={digest}"
    )
if not zipfile.is_zipfile(ZIP_PATH):
    raise RuntimeError("Repository-local RealSign archive is not a valid ZIP file.")

print("Repository-local RealSign archive bytes:", size)
print("Repository-local RealSign archive SHA-256:", digest)

print("Extracting...")
with zipfile.ZipFile(ZIP_PATH) as archive:
    archive.extractall(DATASET_DIR)

# Sanity-check that expected split folders exist somewhere under DATASET_DIR
expected_markers = ("Training", "Testing", "Validation")
found = {m: False for m in expected_markers}
for p in DATASET_DIR.rglob("*"):
    if p.is_dir():
        name = p.name
        for m in expected_markers:
            if m in name:
                found[m] = True
missing = [m for m, ok in found.items() if not ok]
if missing:
    print("WARNING: could not locate folders for:", missing)
    print("Top-level contents of", DATASET_DIR, ":")
    for child in sorted(DATASET_DIR.iterdir())[:30]:
        print(" ", child.name)
else:
    print("Found Training / Validation / Testing folders.")

print("RealSign extraction: PASS")

In [ ]:
# Cell 5 — Extract 126D landmarks (can take 10–40+ minutes depending on CPU)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

out_dir = Path("/content/visionbridge_letter_data")
if out_dir.exists():
    import shutil
    shutil.rmtree(out_dir)

command = [
    str(PYTHON),
    str(REPO / "backend/scripts/prepare_letter_dataset.py"),
    "--input-root", "/content/RealSign",
    "--output-dir", str(out_dir),
    "--validation-ratio", "0.20",
    "--seed", "42",
    "--hand-model-path", "/content/hand_landmarker.task",
]

print("Running landmark preparation (this is the slow step)...", flush=True)
result = subprocess.run(
    command,
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"prepare_letter_dataset.py failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Landmark preparation: PASS")

In [ ]:
# Cell 6 — Validate prepared NPZ contract
from pathlib import Path
import json
import numpy as np

root = Path("/content/visionbridge_letter_data")
metadata = json.loads((root / "labels.json").read_text(encoding="utf-8"))

labels = metadata["labels"]
if labels != list("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError(f"Expected A-Z labels, got {labels}")

print("Labels:", "".join(labels))
print("Split policy:", metadata["split_policy"])

for split in ("train", "val", "test"):
    data = np.load(root / f"{split}.npz")
    x = data["x"]
    y = data["y"]

    if x.ndim != 2 or x.shape[1] != 126:
        raise RuntimeError(f"{split} has invalid shape: {x.shape}")
    if y.ndim != 1 or len(x) != len(y) or len(x) == 0:
        raise RuntimeError(f"{split} has invalid labels")
    if not np.isfinite(x).all():
        raise RuntimeError(f"{split} contains NaN or Inf")

    counts = [int((y == i).sum()) for i in range(26)]
    if any(count == 0 for count in counts):
        raise RuntimeError(f"{split} is missing an A-Z class: {counts}")

    print(f"{split}: samples={len(x)} shape={x.shape}")

duplicate_report = json.loads(
    (root / "duplicate_report.json").read_text(encoding="utf-8")
)
if duplicate_report["train_val_overlap_hashes"]:
    raise RuntimeError("Exact duplicate image hashes leaked between train and validation")

if duplicate_report["train_test_overlap_hashes"] or duplicate_report["val_test_overlap_hashes"]:
    raise RuntimeError(
        "Exact duplicate image hashes still overlap the untouched source test split"
    )

print("Duplicate-leakage contract: PASS")

print("Prepared dataset contract: PASS")


In [ ]:
# Cell 7 — Verify the active VisionBridge V3 architecture contract
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

contract = r'''
import torch
from app.models.letter_model import (
    INPUT_DIM,
    HIDDEN_DIM,
    EMBEDDING_DIM,
    NUM_CLASSES,
    LETTER_LABELS,
    VisionBridgeLetterBaseModel,
)

assert INPUT_DIM == 126, INPUT_DIM
assert HIDDEN_DIM == 128, HIDDEN_DIM
assert EMBEDDING_DIM == 64, EMBEDDING_DIM
assert NUM_CLASSES == 26, NUM_CLASSES
assert tuple(LETTER_LABELS) == tuple("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

model = VisionBridgeLetterBaseModel()
assert model.input_dim == 126
assert model.hidden_dim == 128
assert model.embedding_dim == 64
assert model.num_classes == 26

layers = list(model.encoder.children())
layer_names = [type(layer).__name__ for layer in layers]
expected = ["LayerNorm", "Linear", "GELU", "Dropout", "Linear", "LayerNorm", "GELU"]
assert layer_names == expected, layer_names

sample = torch.zeros(2, 126)
with torch.inference_mode():
    embedding = model.embed(sample)
    logits = model(sample)

assert embedding.shape == (2, 64), embedding.shape
assert logits.shape == (2, 26), logits.shape
assert torch.isfinite(embedding).all()
assert torch.isfinite(logits).all()

print("V3 architecture contract: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("num_classes =", model.num_classes)
print("encoder =", " -> ".join(layer_names))
print("logits_shape =", tuple(logits.shape))
'''

result = subprocess.run(
    [str(PYTHON), "-c", contract],
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"V3 architecture contract failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\n"
        f"stderr:\n{result.stderr}"
    )


In [ ]:
# Cell 8 — Train letter base model (up to 500 epochs; early-stops when all letters hit target)
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

weights_dir = REPO / "backend/app/models/weights"
weights_dir.mkdir(parents=True, exist_ok=True)

command = [
    str(PYTHON),
    "-m",
    "app.training.letter_base",
    "--data-dir", "/content/visionbridge_letter_data",
    "--output", str(weights_dir / "letter_base_model.pt"),
    "--epochs", "500",
    "--batch-size", "128",
    "--lr", "0.001",
    "--weight-decay", "0.0001",
    "--target-class-accuracy", "1.0",
    "--seed", "42",
    "--hidden-dim", "128",
    "--embedding-dim", "64",
    "--dropout", "0.10",
]

print("Starting training...", flush=True)
result = subprocess.run(
    command,
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
# Training can print a lot; still surface full logs on failure
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Training failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )
print("Training command finished successfully.")

In [ ]:
# Cell 9 — Verify checkpoint loads
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")
ENV["MPLBACKEND"] = "Agg"

if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("Training did not create a checkpoint")

check = r'''
import sys
from app.models.letter_model import load_checkpoint

model = load_checkpoint(sys.argv[1])
print("CHECKPOINT LOAD: PASS")
print("input_dim =", model.input_dim)
print("hidden_dim =", model.hidden_dim)
print("embedding_dim =", model.embedding_dim)
print("classes =", model.num_classes)
print("labels =", "".join(model.labels))
'''

result = subprocess.run(
    [str(PYTHON), "-c", check, str(CHECKPOINT)],
    check=False,
    env=ENV,
    text=True,
    capture_output=True,
)
print(result.stdout, end="")
if result.stderr:
    print(result.stderr, end="", file=sys.stderr)
if result.returncode != 0:
    raise RuntimeError(
        f"Checkpoint load failed (exit {result.returncode})\n"
        f"stdout:\n{result.stdout}\nstderr:\n{result.stderr}"
    )

print("Checkpoint bytes:", CHECKPOINT.stat().st_size)
print("Training pipeline: PASS")
print()
print("Download this file from Colab:")
print(" ", CHECKPOINT)

In [ ]:
# Cell 10 — Evaluate the trained checkpoint on the untouched test split
import os
import subprocess
from pathlib import Path

REPO = Path("/content/VisionBridge")
PYTHON = Path("/content/visionbridge_train_env/bin/python")
ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO / "backend")

subprocess.run(
    [
        str(PYTHON),
        "-m",
        "app.training.evaluate_letter_base",
        "--checkpoint",
        str(REPO / "backend/app/models/weights/letter_base_model.pt"),
        "--data-dir",
        "/content/visionbridge_letter_data",
        "--split",
        "test",
        "--output-json",
        "/content/visionbridge_letter_evaluation.json",
    ],
    check=True,
    env=ENV,
)


In [ ]:
# Cell 12 — Build the V3 release evidence manifest
from pathlib import Path
import hashlib
import json

REPO = Path("/content/VisionBridge")
CHECKPOINT = REPO / "backend/app/models/weights/letter_base_model.pt"
EVIDENCE = Path("/content/visionbridge_v3_release_evidence.json")
EVALUATION = Path("/content/visionbridge_letter_evaluation.json")

if not CHECKPOINT.is_file() or CHECKPOINT.stat().st_size == 0:
    raise RuntimeError("V3 checkpoint is missing")
if not EVALUATION.is_file():
    raise RuntimeError("V3 evaluation report is missing")

digest = hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
report = json.loads(EVALUATION.read_text(encoding="utf-8"))
per_letter = report["per_letter_accuracy"]
if set(per_letter) != set("ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    raise RuntimeError("Evaluation report does not contain all A-Z classes")
if any(int(count) <= 0 for count in report["class_counts"].values()):
    raise RuntimeError("Evaluation report contains an empty test class")
if report["model"]["model_version"] != "visionbridge-letter-base-v3":
    raise RuntimeError("Evaluation report is not for the active V3 model")

evidence = {
    "model_version": "visionbridge-letter-base-v3",
    "preprocessing_version": report["model"]["preprocessing_version"],
    "landmark_runtime": report["model"]["landmark_runtime"],
    "training": {
        "epochs": 500,
        "batch_size": 128,
        "learning_rate": 0.001,
        "weight_decay": 0.0001,
        "target_class_accuracy": 1.0,
        "seed": 42,
        "hidden_dim": 128,
        "embedding_dim": 64,
        "dropout": 0.10,
    },
    "dataset_source": {
        "repository_path": "data/raw/RealSign/Dataset.zip",
        "source_repository": "RealSign62/RealSign-Indian-Sign-Language-Dataset",
        "size": 656_689_688,
        "sha256": "008cae248e346b8c31fbbea057fcc3f69c6909d29a88e6bb1fb0369f528de2b5",
    },
    "checkpoint": {
        "path": str(CHECKPOINT),
        "bytes": CHECKPOINT.stat().st_size,
        "sha256": digest,
    },
    "evaluation": report,
    "signer_independent_status": "BLOCKED: verified signer IDs are not exposed by the active source folder structure",
}
EVIDENCE.write_text(json.dumps(evidence, indent=2), encoding="utf-8")

print("V3 release evidence: PASS")
print("checkpoint_sha256 =", digest)
print("test_accuracy =", report["overall_accuracy"])
print("macro_accuracy =", report["macro_accuracy"])
print("worst_letter =", report["worst_letter"], report["worst_letter_accuracy"])
print("evidence =", EVIDENCE)
print("signer_independent_status = BLOCKED pending verified signer metadata")
